In [ ]:
"""
Visualization 0
The script is designed for downstream analysis and visualization for research publication.
The workflow includes:
    - Data loading and preprocessing (reshaping, filtering, flattening)
    - Yield decomposition into observation, trend, and residual
    - Grouping and binning of yield data for visualization
    - Aggregation of model prediction errors for multiple neural network models
    - Generation of feature importance metrics
    - Exporting processed datasets and summary CSVs for figures (Fig.1-Fig.6)
"""

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
import function
import os
for i in range(6):
    os.makedirs(f'../results/book/{i+1}', exist_ok=True)
    os.makedirs(f'../results/book/s{i+1}', exist_ok=True)

### Load datasets.
yield_data = pd.read_csv('../data/yield.csv', engine='python').values[:,1:]
social_data = pd.read_csv('../data/social.csv', engine='python').values[:,1:]
natural_dataset = pd.read_csv('../data/natural.csv', engine='python')
natural_data = function.natural_feature_builder(natural_dataset).values

### Convert data to tensor format and reshape into county × year × feature structure.
yield_data = torch.from_numpy(yield_data.astype('float')).float()
social_data = torch.from_numpy(social_data.astype('float')).float().reshape(-1,22,9)
natural_data = torch.from_numpy(natural_data.astype('float')).float().reshape(22,12,-1,8)[:,:9].permute(2,0,1,3).reshape(654,22,-1)

### Build county index for hierarchical grouping.
### Filter, resolve, and flatten datasets using custom functions.
county_index, counts = np.arange(654).reshape(-1,1), [117, 76, 64, 57, 89, 109, 67, 75]
county_index = np.concatenate([county_index, np.repeat(range(len(counts)), counts).reshape(-1,1)], 1)
yield_data, social_data, natural_data, county_index = function.filter_(yield_data, social_data, natural_data, county_index)
yield_tear, social_tear, natural_tear, county_usles = function.resolve(yield_data, social_data, natural_data, county_index)
yield_data, social_data, natural_data, county_index = function.flatten(yield_data, social_data, natural_data, county_index)
yield_obsed, yield_trend, yield_resid = yield_tear[:,0].reshape(-1,1), yield_tear[:,1].reshape(-1,1), yield_tear[:,2].reshape(-1,1)
social_obsed, natural_obsed = social_tear[0], natural_tear[0]

### Split yield data into early and later periods.
for i in range(2):
    p, index = ['early', 'later'][i], [np.arange(11), np.arange(11, 22)][i]
    exec(f"yield_data_{p} = yield_data[np.isin(county_index[:,2], index)]")
    exec(f"county_index_{p} = county_index[np.isin(county_index[:,2], index)]")

In [ ]:
### Fig.1: Yield distribution and baseline model errors.
yield_csv = pd.DataFrame(np.concatenate([county_index[:,0].reshape(-1,1), yield_data.numpy()], 1), columns = ['county','yield'])
yield_grouped = yield_csv.groupby('county')['yield'].agg(['mean', 'std'])
yield_grouped['cv'] = yield_grouped['std'] / yield_grouped['mean']
yield_grouped.reset_index(inplace=True)
yield_mapping = yield_grouped.sort_values(by='county')[['county', 'mean']]
yield_mapping.to_csv('../results/book/1/yield_mapping.csv', index=False)

yield_grouped['bin'] = pd.cut(yield_grouped['mean'], bins=3, labels=False)
yield_t1 = (yield_grouped['mean'][yield_grouped['bin']==0].max() + yield_grouped['mean'][yield_grouped['bin']==1].min()) /2
yield_t2 = (yield_grouped['mean'][yield_grouped['bin']==1].max() + yield_grouped['mean'][yield_grouped['bin']==2].min()) /2
print(f'yield_t1, yield_t2 = {yield_t1}, {yield_t2}')
yield_grouped = yield_grouped.sort_values(by='bin')
yield_grouped.to_csv('../results/book/1/yield_grouped.csv', index=False)

for model in ['dnn', 'cnn', 'rnn', 'lst', 'gru', 'att']:
    exec(f"error_csv = pd.read_csv('../results/work/1/test_inst_{model}.csv', engine='python')")
    error_grouped = error_csv.groupby('county').agg(['mean']).reset_index()
    error_grouped['bin'] = pd.cut(error_grouped['yield']['mean'], bins=3, labels=False)
    error_grouped = error_grouped.sort_values(by='bin')[['bin', 'mse_base_pret']]
    error_grouped.columns = ['bin', 'mse']
    exec(f"error_grouped.to_csv('../results/book/1/error_grouped_{model}.csv', index=False)")

for model in ['dnn', 'cnn', 'rnn', 'lst', 'gru', 'att']:
    exec(f"error_csv = pd.read_csv('../results/work/1/test_accu_{model}.csv', engine='python')")
    error_csv = error_csv[['Unnamed: 0', 'r2_base', 'mse_base', 'mae_base']]
    error_csv.columns = ['index', 'r2', 'mse', 'mae']
    exec(f"error_csv.to_csv('../results/book/1/error_dataset_{model}.csv', index=False)")
    exec(f"error_csv.to_csv('../results/book/s1/error_dataset_{model}.csv', index=False)")

for model in ['dnn', 'cnn', 'rnn', 'lst', 'gru', 'att']:
    exec(f"error_csv = pd.read_csv('../results/work/1/test_inst_{model}.csv', engine='python')")
    error_grouped = error_csv.groupby('county').agg('mean').reset_index()
    error_grouped['bin'] = pd.cut(error_grouped['yield'], bins=3, labels=False)
    error_grouped = error_grouped[['bin', 'mse_base_pret']].groupby('bin').agg(['mean']).reset_index()
    error_grouped.columns = ['bin', 'mse']
    error_grouped = error_grouped.sort_values(by='bin')
    exec(f"error_grouped.to_csv('../results/book/s1/error_grouped_{model}.csv', index=False)")

for i in range(5):
    for model in ['dnn', 'cnn', 'rnn', 'lst', 'gru', 'att']:
        exec(f"error_csv = pd.read_csv('../results/work/1/test_inst_{model}_kf{i+1}.csv', engine='python')")
        error_grouped = error_csv.groupby('county').agg('mean').reset_index()
        error_grouped['bin'] = pd.cut(error_grouped['yield'], bins=3, labels=False)
        error_grouped = error_grouped[['bin', 'mse_base_pret']].groupby('bin').agg(['mean']).reset_index()
        error_grouped.columns = ['bin', 'mse']
        error_grouped = error_grouped.sort_values(by='bin')
        exec(f"error_grouped.to_csv('../results/book/s1/error_grouped_{model}_kf{i+1}.csv', index=False)")
        
    for model in ['dnn', 'cnn', 'rnn', 'lst', 'gru', 'att']:
        exec(f"error_csv = pd.read_csv('../results/work/1/test_accu_{model}_kf{i+1}.csv', engine='python')")
        error_csv = error_csv[['Unnamed: 0', 'r2_base', 'mse_base', 'mae_base']]
        error_csv.columns = ['index', 'r2', 'mse', 'mae']
        exec(f"error_csv.to_csv('../results/book/s1/error_dataset_{model}_kf{i+1}.csv', index=False)")

social_columns = ['PopS', 'HeaR', 'PuPr', 'TePr', 'FiRv', 'FiEx', 'ReSv', 'InLn']
natural_columns = ['Radn', 'AirT', 'SoTe', 'AirH', 'SoMo', 'Prec', 'SuPr', 'WiSp']
social_csv = pd.DataFrame(np.concatenate([county_index[:,[0,2]], social_data.numpy()], 1), columns = ['county','year'] + social_columns)
natural_csv = pd.DataFrame(np.concatenate([county_index[:,[0,2]], natural_data.reshape(-1,9,8).mean(1).numpy()], 1),
                           columns = ['county','year'] + natural_columns)
feature_csv = pd.concat([natural_csv, social_csv.iloc[:,2:]], 1)
feat_mapping = feature_csv.groupby('county').agg('mean').reset_index()[['county'] + natural_columns + social_columns]
feat_mapping.to_csv('../results/book/s1/feat_mapping.csv', index=False)

feat_grouped = feature_csv.iloc[:,1:].groupby('year').agg('mean').reset_index()
feat_grouped.to_csv('../results/book/s1/feat_grouped.csv', index=False)

In [ ]:
### Fig.2: Feature importance analysis.
yield_columns = ['YO', 'TY', 'DY']
social_columns = ['PopS', 'HeaR', 'PuPr', 'TePr', 'FiRv', 'FiEx', 'ReSv', 'InLn']
natural_columns = ['Radn', 'AirT', 'SoTe', 'AirH', 'SoMo', 'Prec', 'SuPr', 'WiSp']
yield_csv = pd.DataFrame(np.concatenate([county_index[:,2].reshape(-1,1), yield_data.numpy()], 1), columns = ['year', 'yield'])
yield_yo = yield_csv.groupby('year').agg('mean').values
yield_ty = (yield_yo + np.concatenate([yield_yo[0].reshape(1,-1), yield_yo[:-1]])) / 2
yield_dy = yield_yo
yield_grouped = pd.DataFrame(np.concatenate([yield_yo, yield_ty, yield_dy], 1), columns = yield_columns)
yield_grouped.to_csv('../results/book/2/yield_grouped.csv', index=False)

yield_csv = pd.DataFrame(np.concatenate([county_usles[:,[0,2]], yield_obsed.numpy(), yield_trend.numpy(), yield_resid.numpy()], 1),
                         columns = ['county', 'year', 'YO', 'TY', 'DY'])
social_csv = pd.DataFrame(np.concatenate([county_usles[:,[0,2]], social_obsed.numpy()], 1), columns = ['county','year'] + social_columns)
natural_csv = pd.DataFrame(np.concatenate([county_usles[:,[0,2]], natural_obsed.reshape(-1,9,8).mean(1).numpy()], 1),
                           columns = ['county','year'] + natural_columns)
for i in ['yield', 'social', 'natural']:
    exec(f"dataset, columns = pd.DataFrame([]), {i}_columns")
    for j in columns:
        exec(f"data_scaled = {i}_csv.pivot(index = 'year', columns = 'county', values = '{j}')")
        data_scaled = StandardScaler().fit_transform(data_scaled.values)
        data_scaled = torch.from_numpy(data_scaled.reshape(-1)).float()
        index_notnan = np.arange(data_scaled.size(0))[~torch.isnan(data_scaled)]
        data_scaled = data_scaled[index_notnan]
        exec(f"dataset['{j}'] = data_scaled")
    exec(f"{i}_dataset = dataset")
yield_dataset.to_csv('../results/book/2/yield_dataset.csv', index=False)
feat_dataset = pd.concat([social_dataset, natural_dataset], 1)
feat_dataset.to_csv('../results/book/2/feat_dataset.csv', index=False)

for model in ['dnn', 'cnn', 'rnn', 'lst', 'gru', 'att']:
    feat = []
    for i in ['obsvd', 'resid']:
        exec(f"feat_ig = pd.read_csv('../results/work/2/{i}_ig_{model}.csv', engine='python').abs().values[:,2:]")
        exec(f"feat_gs = pd.read_csv('../results/work/2/{i}_gs_{model}.csv', engine='python').abs().values[:,2:]")
        feat_mean = np.concatenate([[feat_ig], [feat_gs]]).mean(0).mean(0)
        feat_mean = np.concatenate([feat_mean[:8], feat_mean[8:].reshape(9,8).mean(0)])
        feat_mean = feat_mean / feat_mean.sum()
        feat.append(feat_mean)
    feat_csv = pd.DataFrame(np.array(feat).T, index = social_columns + natural_columns, columns = ['obsvd', 'resid'])
    lsfg_csv = pd.DataFrame(feat_csv.values[:8].sum(0).reshape(1,2), index = ['LSFG'], columns = ['obsvd', 'resid'])
    nefg_csv = pd.DataFrame(feat_csv.values[8:].sum(0).reshape(1,2), index = ['NEFG'], columns = ['obsvd', 'resid'])
    feat_book_csv = pd.concat([feat_csv.iloc[:8], lsfg_csv, feat_csv.iloc[8:], nefg_csv], 0)
    feat_book_csv = feat_book_csv.reset_index()
    exec(f"feat_book_csv.to_csv('../results/book/2/feature_book_{model}.csv', index=False)")
    feat_csv = feat_csv.reindex(natural_columns + social_columns[::-1]).reset_index()
    exec(f"feat_csv.to_csv('../results/book/2/feat_imptnce_{model}.csv', index=False)")

county_usles_csv = pd.DataFrame(county_usles, columns = ['county', 'province', 'year'])
province_index = county_usles_csv.pivot(index = 'year', columns = 'county', values = 'province').values
province_index = torch.from_numpy(province_index.reshape(-1)).float()
index_notnan = np.arange(province_index.size(0))[~torch.isnan(province_index)]
province_index = province_index[index_notnan].int().numpy()
for i in range(8):
    p = ['hebei', 'shanxi', 'jiangsu', 'anhui', 'shandong', 'henan', 'hubei', 'shaanxi'][i]
    exec(f"yield_dataset_{p}, feat_dataset_{p} = yield_dataset[province_index==i], feat_dataset[province_index==i]")
    exec(f"yield_dataset_{p}.to_csv('../results/book/s2/{p}_yield_dataset.csv', index=False)")
    exec(f"feat_dataset_{p}.to_csv('../results/book/s2/{p}_feat_dataset.csv', index=False)")

    for model in ['dnn', 'cnn', 'rnn', 'lst', 'gru', 'att']:
        feat = []
        for j in ['obsvd', 'resid']:
            exec(f"feat_ig = pd.read_csv('../results/work/4/{p}_{j}_ig_{model}.csv', engine='python').abs().values[:,2:]")
            exec(f"feat_gs = pd.read_csv('../results/work/4/{p}_{j}_gs_{model}.csv', engine='python').abs().values[:,2:]")
            feat_mean = np.concatenate([[feat_ig], [feat_gs]]).mean(0).mean(0)
            feat_mean = np.concatenate([feat_mean[:8], feat_mean[8:].reshape(9,8).mean(0)])
            feat_mean = feat_mean / feat_mean.sum()
            feat.append(feat_mean)
        feat_csv = pd.DataFrame(np.array(feat).T, index = social_columns + natural_columns, columns = ['obsvd', 'resid'])
        feat_csv = feat_csv.reindex(natural_columns + social_columns[::-1]).reset_index()
        exec(f"feat_csv.to_csv('../results/book/s2/{p}_feat_imptnce_{model}.csv', index=False)")

In [ ]:
### Fig.3: Yield-bin error analysis.
error_csv = pd.read_csv('../results/work/1/test_inst_dnn.csv', engine='python')
error_csv['bin'] = pd.cut(error_csv['yield'], bins=3, labels=False)
error_grouped = error_csv.groupby('bin').agg(['count','mean']).reset_index()
count = error_grouped.xs('count', level=1, axis=1)[['yield']]
error = error_grouped.xs('mean', level=1, axis=1)[['mse_base_pret']]
error_grouped = pd.concat([error_grouped[['bin']], count, error], 1)
error_grouped.columns = ['bin', 'yield', 'mse']
error_grouped.to_csv('../results/book/3/error_grouped.csv', index=False)

yield_csv = pd.DataFrame(np.concatenate([county_index[:,1].reshape(-1,1), yield_data.numpy()], 1), columns = ['province','yield'])
yield_csv.to_csv('../results/book/3/yield_dataset.csv', index=False)

bins = range(0, 9001, 500)
bin_centers = [(bins[i] + bins[i+1]) // 2 for i in range(len(bins)-1)]
for model in ['dnn', 'cnn', 'rnn', 'lst', 'gru', 'att']:
    exec(f"error_csv = pd.read_csv('../results/work/1/test_inst_{model}.csv', engine='python')")
    error_csv['bin'] = pd.cut(error_csv['yield'], bins=bins, labels=bin_centers)
    error_grouped = error_csv.groupby('bin').agg(['count','mean']).reset_index()
    count = error_grouped.xs('count', level=1, axis=1)[['yield']]
    error = error_grouped.xs('mean', level=1, axis=1)[['mse_base_pret', 'mse_addi_pret']]
    error_grouped = pd.concat([error_grouped[['bin']], count, error], 1)
    error_grouped.columns = ['bin', 'count', 'mse_base', 'mse_addi']
    exec(f"error_grouped.to_csv('../results/book/3/error_grouped_{model}.csv', index=False)")

for model in ['dnn', 'cnn', 'rnn', 'lst', 'gru', 'att']:
    exec(f"error_csv = pd.read_csv('../results/work/1/test_accu_{model}.csv', engine='python')")
    error_csv.columns = ['index', 'r2_base', 'mse_base', 'mae_base', 'r2_addi', 'mse_addi', 'mae_addi']
    exec(f"error_csv.to_csv('../results/book/3/error_dataset_{model}.csv', index=False)")

for model in ['dnn', 'cnn', 'rnn', 'lst', 'gru', 'att']:
    exec(f"error_csv = pd.read_csv('../results/work/1/test_inst_{model}.csv', engine='python')")
    error_csv['bin'] = pd.cut(error_csv['yield'], bins=bins, labels=bin_centers)
    error_grouped = error_csv.groupby('bin').agg(['count','mean']).reset_index()
    count = error_grouped.xs('count', level=1, axis=1)[['yield']]
    error = error_grouped.xs('mean', level=1, axis=1)[['mse_base_pret', 'mse_addi_pret']]
    error_grouped = pd.concat([error_grouped[['bin']], count, error], 1)
    error_grouped.columns = ['bin', 'count', 'mse_base', 'mse_addi']
    exec(f"error_grouped.to_csv('../results/book/s3/error_grouped_{model}.csv', index=False)")
    
    for i in range(5):
        exec(f"error_csv = pd.read_csv('../results/work/1/test_inst_{model}_kf{i+1}.csv', engine='python')")
        error_csv['bin'] = pd.cut(error_csv['yield'], bins=bins, labels=bin_centers)
        error_grouped = error_csv.groupby('bin').agg(['count','mean']).reset_index()
        count = error_grouped.xs('count', level=1, axis=1)[['yield']]
        error = error_grouped.xs('mean', level=1, axis=1)[['mse_base_pret', 'mse_addi_pret']]
        error_grouped = pd.concat([error_grouped[['bin']], count, error], 1)
        error_grouped.columns = ['bin', 'count', 'mse_base', 'mse_addi']
        exec(f"error_grouped.to_csv('../results/book/s3/error_grouped_{model}_kf{i+1}.csv', index=False)")

In [ ]:
### Fig.4: FGAA training dynamics.
yield_csv = pd.DataFrame(np.concatenate([county_index[:,1].reshape(-1,1), yield_data.numpy()], 1), columns = ['province','yield'])
yield_csv.to_csv('../results/book/4/yield_dataset.csv', index=False)

bins, error_grouped_gather = range(0, 9001, 500), []
bin_centers = [(bins[i] + bins[i+1]) // 2 for i in range(len(bins)-1)]
for i in range(5):
    exec(f"error_csv = pd.read_csv('../results/work/1/train_inst_dnn_kf{i+1}.csv', engine='python')")
    error_csv['bin'] = pd.cut(error_csv['yield'], bins=bins, labels=bin_centers)
    error_grouped = error_csv.groupby('bin').agg(['count','mean']).reset_index()
    error_grouped_gather.append([error_grouped.values])
error_grouped_gather = np.concatenate(error_grouped_gather, 0)
error_grouped_gather = np.apply_along_axis(lambda x: np.nanmean(x) if np.any(~np.isnan(x)) else np.nan, 0, error_grouped_gather)
error_grouped_gather = pd.DataFrame(error_grouped_gather, columns = error_grouped.columns)
count = error_grouped_gather.xs('count', level=1, axis=1)[['yield']]
error = error_grouped_gather.xs('mean', level=1, axis=1)[['mse_base_init', 'mse_base_pret', 'scores_base_pret', 'mse_base_fine']]
error_grouped_gather = pd.concat([error_grouped_gather[['bin']], count, error], 1)
error_grouped_gather.columns = ['bin', 'count', 'mse_init', 'mse_pret', 'scores', 'mse_fine']
error_grouped_gather.to_csv('../results/book/4/error_grouped1_train.csv', index=False)

error_csv = pd.read_csv('../results/work/1/test_inst_dnn.csv', engine='python')
error_csv['bin'] = pd.cut(error_csv['yield'], bins=bins, labels=bin_centers)
error_grouped = error_csv.groupby('bin').agg(['count','mean']).reset_index()
count = error_grouped.xs('count', level=1, axis=1)[['yield']]
error = error_grouped.xs('mean', level=1, axis=1)[['mse_base_init', 'mse_base_pret', 'mse_base_fine']]
error_grouped = pd.concat([error_grouped[['bin']], count, error], 1)
error_grouped.columns = ['bin', 'count', 'mse_init', 'mse_pret', 'mse_fine']
error_grouped.to_csv('../results/book/4/error_grouped1_test.csv', index=False)

for i in ['train', 'test']:
    exec(f"error_csv = pd.read_csv('../results/work/1/{i}_accu_dnn.csv', engine='python').iloc[:,:4]")
    error_csv.columns = ['index', 'r2', 'mse', 'mae']
    exec(f"error_csv.to_csv('../results/book/4/error_dataset_{i}.csv', index=False)")

error_grouped_gather = []
for i in range(5):
    exec(f"error_csv = pd.read_csv('../results/work/1/train_inst_dnn_kf{i+1}.csv', engine='python')")
    error_csv['bin'] = pd.cut(error_csv['yield'], bins=3, labels=False)
    error_grouped = error_csv.groupby('bin').agg('mean').reset_index()
    error_grouped_gather.append([error_grouped.values])
error_grouped_gather = np.concatenate(error_grouped_gather, 0)
error_grouped_gather = np.apply_along_axis(lambda x: np.nanmean(x) if np.any(~np.isnan(x)) else np.nan, 0, error_grouped_gather)
error_grouped_gather = pd.DataFrame(error_grouped_gather, columns = error_grouped.columns)
error_grouped_gather = error_grouped_gather[['mse_base_init', 'mse_base_pret', 'mse_base_fine']].T
error_grouped_gather.columns, error_grouped_gather.index = ['low', 'medium', 'high'], ['mse_init', 'mse_pret', 'mse_fine']
error_grouped_gather.to_csv('../results/book/4/error_grouped2_train.csv')

error_csv = pd.read_csv('../results/work/1/test_inst_dnn.csv', engine='python')
error_csv['bin'] = pd.cut(error_csv['yield'], bins=3, labels=False)
error_grouped = error_csv.groupby('bin').agg('mean').reset_index()
error_grouped = error_grouped[['mse_base_init', 'mse_base_pret', 'mse_base_fine']].T
error_grouped.columns, error_grouped.index = ['low', 'medium', 'high'], ['mse_init', 'mse_pret', 'mse_fine']
error_grouped.to_csv('../results/book/4/error_grouped2_test.csv')

for p in ['hebei', 'shanxi', 'jiangsu', 'anhui', 'shandong', 'henan', 'hubei', 'shaanxi']:
    bins, error_grouped_gather = range(0, 9001, 500), []
    bin_centers = [(bins[i] + bins[i+1]) // 2 for i in range(len(bins)-1)]
    for i in range(5):
        exec(f"error_csv = pd.read_csv('../results/work/3/{p}_train_inst_dnn_kf{i+1}.csv', engine='python')")
        error_csv['bin'] = pd.cut(error_csv['yield'], bins=bins, labels=bin_centers)
        error_grouped = error_csv.groupby('bin').agg(['count','mean']).reset_index()
        error_grouped_gather.append([error_grouped.values])
    error_grouped_gather = np.concatenate(error_grouped_gather, 0)
    error_grouped_gather = np.apply_along_axis(lambda x: np.nanmean(x) if np.any(~np.isnan(x)) else np.nan, 0, error_grouped_gather)
    error_grouped_gather = pd.DataFrame(error_grouped_gather, columns = error_grouped.columns)
    count = error_grouped_gather.xs('count', level=1, axis=1)[['yield']]
    score = error_grouped_gather.xs('mean', level=1, axis=1)[['scores_base_pret', 'scores_addi_pret']]
    score_grouped_gather = pd.concat([error_grouped_gather[['bin']], count, score], 1)
    score_grouped_gather.columns = ['bin', 'count', 'scores_base', 'scores_addi']
    exec(f"score_grouped_gather.to_csv('../results/book/4/score_grouped_{p}.csv', index=False)")

for model in ['cnn', 'rnn', 'lst', 'gru', 'att']:
    bins, error_grouped_gather = range(0, 9001, 500), []
    bin_centers = [(bins[i] + bins[i+1]) // 2 for i in range(len(bins)-1)]
    for i in range(5):
        exec(f"error_csv = pd.read_csv('../results/work/1/train_inst_{model}_kf{i+1}.csv', engine='python')")
        error_csv['bin'] = pd.cut(error_csv['yield'], bins=bins, labels=bin_centers)
        error_grouped = error_csv.groupby('bin').agg(['count','mean']).reset_index()
        error_grouped_gather.append([error_grouped.values])
    error_grouped_gather = np.concatenate(error_grouped_gather, 0)
    error_grouped_gather = np.apply_along_axis(lambda x: np.nanmean(x) if np.any(~np.isnan(x)) else np.nan, 0, error_grouped_gather)
    error_grouped_gather = pd.DataFrame(error_grouped_gather, columns = error_grouped.columns)
    count = error_grouped_gather.xs('count', level=1, axis=1)[['yield']]
    error = error_grouped_gather.xs('mean', level=1, axis=1)[['mse_base_init', 'mse_base_pret', 'mse_base_fine']]
    error_grouped_gather = pd.concat([error_grouped_gather[['bin']], count, error], 1)
    error_grouped_gather.columns = ['bin', 'count', 'mse_init', 'mse_pret', 'mse_fine']
    exec(f"error_grouped_gather.to_csv('../results/book/s4/{model}_error_grouped_train.csv', index=False)")

    exec(f"error_csv = pd.read_csv('../results/work/1/test_inst_{model}.csv', engine='python')")
    error_csv['bin'] = pd.cut(error_csv['yield'], bins=bins, labels=bin_centers)
    error_grouped = error_csv.groupby('bin').agg(['count','mean']).reset_index()
    count = error_grouped.xs('count', level=1, axis=1)[['yield']]
    error = error_grouped.xs('mean', level=1, axis=1)[['mse_base_init', 'mse_base_pret', 'mse_base_fine']]
    error_grouped = pd.concat([error_grouped[['bin']], count, error], 1)
    error_grouped.columns = ['bin', 'count', 'mse_init', 'mse_pret', 'mse_fine']
    exec(f"error_grouped.to_csv('../results/book/s4/{model}_error_grouped_test.csv', index=False)")

    for i in ['train', 'test']:
        exec(f"error_csv = pd.read_csv('../results/work/1/{i}_accu_{model}.csv', engine='python').iloc[:,:4]")
        error_csv.columns = ['index', 'r2', 'mse', 'mae']
        exec(f"error_csv.to_csv('../results/book/s4/{model}_error_dataset_{i}.csv', index=False)")

    for p in ['hebei', 'shanxi', 'jiangsu', 'anhui', 'shandong', 'henan', 'hubei', 'shaanxi']:
        bins, error_grouped_gather = range(0, 9001, 500), []
        bin_centers = [(bins[i] + bins[i+1]) // 2 for i in range(len(bins)-1)]
        for i in range(5):
            exec(f"error_csv = pd.read_csv('../results/work/3/{p}_train_inst_{model}_kf{i+1}.csv', engine='python')")
            error_csv['bin'] = pd.cut(error_csv['yield'], bins=bins, labels=bin_centers)
            error_grouped = error_csv.groupby('bin').agg(['count','mean']).reset_index()
            error_grouped_gather.append([error_grouped.values])
        error_grouped_gather = np.concatenate(error_grouped_gather, 0)
        error_grouped_gather = np.apply_along_axis(lambda x: np.nanmean(x) if np.any(~np.isnan(x)) else np.nan, 0, error_grouped_gather)
        error_grouped_gather = pd.DataFrame(error_grouped_gather, columns = error_grouped.columns)
        count = error_grouped_gather.xs('count', level=1, axis=1)[['yield']]
        score = error_grouped_gather.xs('mean', level=1, axis=1)[['scores_base_pret', 'scores_addi_pret']]
        score_grouped_gather = pd.concat([error_grouped_gather[['bin']], count, score], 1)
        score_grouped_gather.columns = ['bin', 'count', 'scores_base', 'scores_addi']
        exec(f"score_grouped_gather.to_csv('../results/book/s4/{model}_score_grouped_{p}.csv', index=False)")

In [ ]:
### Fig.5: Performance improvement after the Fit–Generalization Adaptive Attention (FGAA) algorithm.
for model in ['dnn', 'cnn', 'rnn', 'lst', 'gru', 'att']:
    exec(f"error_csv = pd.read_csv('../results/work/1/test_inst_{model}.csv', engine='python')")
    error_grouped = error_csv.groupby('county').agg(['mean']).reset_index()
    error_grouped['bin'] = pd.cut(error_grouped['yield']['mean'], bins=3, labels=False)
    error_grouped = error_grouped.sort_values(by='bin')[['bin', 'mse_base_pret', 'mse_addi_fine']]
    error_grouped.columns = ['bin', 'mse_base', 'mse_prop']
    exec(f"error_grouped.to_csv('../results/book/5/error_grouped_{model}.csv', index=False)")

for model in ['dnn', 'cnn', 'rnn', 'lst', 'gru', 'att']:
    exec(f"error_csv = pd.read_csv('../results/work/1/test_accu_{model}.csv', engine='python')")
    error_csv.columns = ['index', 'r2_base', 'mse_base', 'mae_base', 'r2_prop', 'mse_prop', 'mae_prop']
    exec(f"error_csv.to_csv('../results/book/5/error_dataset_{model}.csv', index=False)")

for model in ['dnn', 'cnn', 'rnn', 'lst', 'gru', 'att']:
    exec(f"error_csv = pd.read_csv('../results/work/1/test_inst_{model}.csv', engine='python')")
    error_grouped = error_csv.groupby('county').agg('mean').reset_index()
    error_grouped = error_grouped.sort_values(by='county')[['county', 'mse_base_pret', 'mse_addi_fine']]
    county_list = pd.DataFrame({'county': sorted(set(county_index[:,0]))})
    error_grouped = county_list.merge(error_grouped, on='county', how='left')
    error_grouped.columns = ['county', 'mse_base', 'mse_prop']
    for i in ['base', 'prop']:
        exec(f"error_grouped['bin_{i}'] = pd.qcut(error_grouped['mse_{i}'], q=2, labels=False)")
        exec(f"error_grouped['bin_{i}'] = error_grouped['bin_{i}'].fillna(2)")
    error_grouped = error_grouped[['county', 'bin_base', 'bin_prop']]
    exec(f"error_grouped.to_csv('../results/book/5/error_mapping_{model}.csv', index=False)")

for model in ['dnn', 'cnn', 'rnn', 'lst', 'gru', 'att']:
    exec(f"error_csv = pd.read_csv('../results/work/1/test_accu_{model}.csv', engine='python')")
    error_csv.columns = ['index', 'r2_base', 'mse_base', 'mae_base', 'r2_addi', 'mse_addi', 'mae_addi']
    exec(f"error_csv.to_csv('../results/book/s5/error_dataset_{model}.csv', index=False)")

    exec(f"error_csv = pd.read_csv('../results/work/1/test_inst_{model}.csv', engine='python')")
    error_grouped = error_csv.groupby('county').agg('mean').reset_index()
    error_grouped['bin'] = pd.cut(error_grouped['yield'], bins=3, labels=False)
    error_grouped = error_grouped[['bin', 'mse_base_pret', 'mse_addi_fine']].groupby('bin').agg(['mean']).reset_index()
    error_grouped.columns = ['bin', 'mse_base', 'mse_prop']
    error_grouped = error_grouped.sort_values(by='bin')
    exec(f"error_grouped.to_csv('../results/book/s5/error_grouped_{model}.csv', index=False)")

for i in range(5):
    for model in ['dnn', 'cnn', 'rnn', 'lst', 'gru', 'att']:
        exec(f"error_csv = pd.read_csv('../results/work/1/test_inst_{model}_kf{i+1}.csv', engine='python')")
        error_grouped = error_csv.groupby('county').agg('mean').reset_index()
        error_grouped['bin'] = pd.cut(error_grouped['yield'], bins=3, labels=False)
        error_grouped = error_grouped[['bin', 'mse_base_pret', 'mse_addi_fine']].groupby('bin').agg(['mean']).reset_index()
        error_grouped.columns = ['bin', 'mse_base', 'mse_prop']
        error_grouped = error_grouped.sort_values(by='bin')
        exec(f"error_grouped.to_csv('../results/book/s5/error_grouped_{model}_kf{i+1}.csv', index=False)")

    for model in ['dnn', 'cnn', 'rnn', 'lst', 'gru', 'att']:
        exec(f"error_csv = pd.read_csv('../results/work/1/test_accu_{model}_kf{i+1}.csv', engine='python')")
        error_csv.columns = ['index', 'r2_base', 'mse_base', 'mae_base', 'r2_addi', 'mse_addi', 'mae_addi']
        exec(f"error_csv.to_csv('../results/book/s5/error_dataset_{model}_kf{i+1}.csv', index=False)")

for p in ['hebei', 'shanxi', 'jiangsu', 'anhui', 'shandong', 'henan', 'hubei', 'shaanxi']:
    for model in ['dnn', 'cnn', 'rnn', 'lst', 'gru', 'att']:
        exec(f"error_csv = pd.read_csv('../results/work/3/{p}_test_accu_{model}.csv', engine='python')")
        error_csv.columns = ['index', 'r2_base', 'mse_base', 'mae_base', 'r2_addi', 'mse_addi', 'mae_addi']
        exec(f"error_csv.to_csv('../results/book/s5/{p}_error_dataset_{model}.csv', index=False)")

        exec(f"error_csv = pd.read_csv('../results/work/3/{p}_test_inst_{model}.csv', engine='python')")
        error_grouped = error_csv.groupby('county').agg('mean').reset_index()
        error_grouped['bin'] = pd.cut(error_grouped['yield'], bins=3, labels=False)
        error_grouped = error_grouped[['bin', 'mse_base_pret', 'mse_addi_fine']].groupby('bin').agg(['mean']).reset_index()
        error_grouped.columns = ['bin', 'mse_base', 'mse_prop']
        error_grouped = error_grouped.sort_values(by='bin')
        exec(f"error_grouped.to_csv('../results/book/s5/{p}_error_grouped_{model}.csv', index=False)")

In [ ]:
### Fig.6: Early vs later period comparison.
bins = range(0, 9001, 500)
bin_centers = [(bins[i] + bins[i+1]) // 2 for i in range(len(bins)-1)]
for p in ['early', 'later']:
    exec(f"yield_csv = pd.DataFrame(np.concatenate([county_index_{p}[:,0].reshape(-1,1), yield_data_{p}.numpy()], 1))")
    yield_csv.columns = ['county', 'yield']
    yield_grouped = yield_csv.groupby('county')['yield'].agg(['mean', 'std'])
    yield_grouped['cv'] = yield_grouped['std'] / yield_grouped['mean']
    yield_grouped.reset_index(inplace=True)
    yield_grouped['bin'] = pd.cut(yield_grouped['mean'], bins=3, labels=False)
    yield_t1 = (yield_grouped['mean'][yield_grouped['bin']==0].max() + yield_grouped['mean'][yield_grouped['bin']==1].min()) /2
    yield_t2 = (yield_grouped['mean'][yield_grouped['bin']==1].max() + yield_grouped['mean'][yield_grouped['bin']==2].min()) /2
    exec(f"print('{p}_t1, {p}_t2 = {yield_t1}, {yield_t2}')")
    yield_grouped = yield_grouped.sort_values(by='bin')
    exec(f"yield_grouped.to_csv('../results/book/6/yield_grouped_{p}.csv', index=False)")

    exec(f"yield_csv = pd.DataFrame(np.concatenate([county_index_{p}[:,0].reshape(-1,1), yield_data_{p}.numpy()], 1))")
    yield_csv.columns = ['county', 'yield']
    yield_grouped = yield_csv.groupby('county').agg('mean').reset_index()
    yield_grouped = yield_grouped.sort_values(by='county')
    exec(f"county_{p} = yield_grouped['county']")
    exec(f"yield_grouped.to_csv('../results/book/6/yield_mapping_{p}.csv', index=False)")

    feat = []
    for i in ['obsvd', 'resid']:
        exec(f"feat_ig = pd.read_csv('../results/work/6/{p}_{i}_ig_dnn.csv', engine='python').abs().values[:,2:]")
        exec(f"feat_gs = pd.read_csv('../results/work/6/{p}_{i}_gs_dnn.csv', engine='python').abs().values[:,2:]")
        feat_mean = np.concatenate([[feat_ig], [feat_gs]]).mean(0).mean(0)
        feat_mean = np.concatenate([feat_mean[:8], feat_mean[8:].reshape(9,8).mean(0)])
        feat_mean = feat_mean / feat_mean.sum()
        feat.append(feat_mean)
    feat_csv = pd.DataFrame(np.array(feat).T, index = social_columns + natural_columns, columns = ['obsvd','resid'])
    feat_csv = feat_csv.reindex(natural_columns + social_columns[::-1]).reset_index()
    exec(f"feat_csv.to_csv('../results/book/6/feat_imptnce_{p}.csv', index=False)")

bins = range(0, 9001, 500)
bin_centers = [(bins[i] + bins[i+1]) // 2 for i in range(len(bins)-1)]
for p in ['early', 'later']:
    error_grouped_gather = []
    for i in range(5):
        exec(f"error_csv = pd.read_csv('../results/work/5/{p}_train_inst_dnn_kf{i+1}.csv', engine='python')")
        error_csv['bin'] = pd.cut(error_csv['yield'], bins=bins, labels=bin_centers)
        error_grouped = error_csv.groupby('bin').agg(['count','mean']).reset_index()
        error_grouped_gather.append([error_grouped.values])
    error_grouped_gather = np.concatenate(error_grouped_gather, 0)
    error_grouped_gather = np.apply_along_axis(lambda x: np.nanmean(x) if np.any(~np.isnan(x)) else np.nan, 0, error_grouped_gather)
    error_grouped_gather = pd.DataFrame(error_grouped_gather, columns = error_grouped.columns)
    count = error_grouped_gather.xs('count', level=1, axis=1)[['yield']]
    error = error_grouped_gather.xs('mean', level=1, axis=1)[['mse_addi_pret', 'scores_addi_pret', 'mse_addi_fine']]
    error_grouped_gather = pd.concat([error_grouped_gather[['bin']], count, error], 1)
    error_grouped_gather.columns = ['bin', 'count', 'mse_pret', 'scores', 'mse_fine']
    exec(f"error_grouped_gather.to_csv('../results/book/6/error_group_train_{p}.csv', index=False)")

for p in ['early', 'later']:
    exec(f"error_csv = pd.read_csv('../results/work/5/{p}_test_inst_dnn.csv', engine='python')")
    error_grouped = error_csv.groupby('county').agg(['mean']).reset_index()
    error_grouped['bin'] = pd.cut(error_grouped['yield']['mean'], bins=3, labels=False)
    error_grouped = error_grouped.sort_values(by='bin')[['bin', 'mse_base_pret', 'mse_addi_fine']]
    error_grouped.columns = ['bin', 'mse_base', 'mse_prop']
    exec(f"error_grouped.to_csv('../results/book/6/error_group_test_{p}.csv', index=False)")

for p in ['early', 'later']:
    exec(f"error_csv = pd.read_csv('../results/work/5/{p}_test_accu_dnn.csv', engine='python')")
    error_csv.columns = ['index', 'r2_base', 'mse_base', 'mae_base', 'r2_prop', 'mse_prop', 'mae_prop']
    exec(f"error_csv.to_csv('../results/book/6/error_dataset_{p}.csv', index=False)")

for p in ['early', 'later']:
    exec(f"error_csv = pd.read_csv('../results/work/5/{p}_test_inst_dnn.csv', engine='python')")
    error_grouped = error_csv.groupby('county').agg('mean').reset_index()
    error_grouped = error_grouped.sort_values(by='county')[['county', 'mse_base_pret', 'mse_addi_fine']]
    exec(f"error_grouped = pd.DataFrame(county_{p}).merge(error_grouped, on='county', how='left')")
    error_grouped.columns = ['county', 'mse_base', 'mse_prop']
    for i in ['base', 'prop']:
        exec(f"error_grouped['bin_{i}'] = pd.qcut(error_grouped['mse_{i}'], q=2, labels=False)")
        exec(f"error_grouped['bin_{i}'] = error_grouped['bin_{i}'].fillna(2)")
    error_grouped = error_grouped[['county', 'bin_base', 'bin_prop']]
    exec(f"error_grouped.to_csv('../results/book/6/error_mapping_{p}.csv', index=False)")

for model in ['cnn', 'rnn', 'lst', 'gru', 'att']:
    for p in ['early', 'later']:
        feat = []
        for i in ['obsvd', 'resid']:
            exec(f"feat_ig = pd.read_csv('../results/work/6/{p}_{i}_ig_{model}.csv', engine='python').abs().values[:,2:]")
            exec(f"feat_gs = pd.read_csv('../results/work/6/{p}_{i}_gs_{model}.csv', engine='python').abs().values[:,2:]")
            feat_mean = np.concatenate([[feat_ig], [feat_gs]]).mean(0).mean(0)
            feat_mean = np.concatenate([feat_mean[:8], feat_mean[8:].reshape(9,8).mean(0)])
            feat_mean = feat_mean / feat_mean.sum()
            feat.append(feat_mean)
        feat_csv = pd.DataFrame(np.array(feat).T, index = social_columns + natural_columns, columns = ['obsvd', 'resid'])
        feat_csv = feat_csv.reindex(natural_columns + social_columns[::-1]).reset_index()
        exec(f"feat_csv.to_csv('../results/book/s6/{model}_{p}_feat_imptnce.csv', index=False)")

        error_grouped_gather = []
        for i in range(5):
            exec(f"error_csv = pd.read_csv('../results/work/5/{p}_train_inst_{model}_kf{i+1}.csv', engine='python')")
            error_csv['bin'] = pd.cut(error_csv['yield'], bins=bins, labels=bin_centers)
            error_grouped = error_csv.groupby('bin').agg(['count','mean']).reset_index()
            error_grouped_gather.append([error_grouped.values])
        error_grouped_gather = np.concatenate(error_grouped_gather, 0)
        error_grouped_gather = np.apply_along_axis(lambda x: np.nanmean(x) if np.any(~np.isnan(x)) else np.nan, 0, error_grouped_gather)
        error_grouped_gather = pd.DataFrame(error_grouped_gather, columns = error_grouped.columns)
        count = error_grouped_gather.xs('count', level=1, axis=1)[['yield']]
        error = error_grouped_gather.xs('mean', level=1, axis=1)[['mse_addi_pret', 'scores_addi_pret', 'mse_addi_fine']]
        error_grouped_gather = pd.concat([error_grouped_gather[['bin']], count, error], 1)
        error_grouped_gather.columns = ['bin', 'count', 'mse_pret', 'scores', 'mse_fine']
        exec(f"error_grouped_gather.to_csv('../results/book/s6/{model}_{p}_error_train.csv', index=False)")

        exec(f"error_csv = pd.read_csv('../results/work/5/{p}_test_inst_{model}.csv', engine='python')")
        error_grouped = error_csv.groupby('county').agg(['mean']).reset_index()
        error_grouped['bin'] = pd.cut(error_grouped['yield']['mean'], bins=3, labels=False)
        error_grouped = error_grouped.sort_values(by='bin')[['bin', 'mse_base_pret', 'mse_addi_fine']]
        error_grouped.columns = ['bin', 'mse_base', 'mse_prop']
        exec(f"error_grouped.to_csv('../results/book/s6/{model}_{p}_error_test.csv', index=False)")

        exec(f"error_csv = pd.read_csv('../results/work/5/{p}_test_accu_{model}.csv', engine='python')")
        error_csv.columns = ['index', 'r2_base', 'mse_base', 'mae_base', 'r2_prop', 'mse_prop', 'mae_prop']
        exec(f"error_csv.to_csv('../results/book/s6/{model}_{p}_error_dataset.csv', index=False)")

        exec(f"error_csv = pd.read_csv('../results/work/5/{p}_test_inst_{model}.csv', engine='python')")
        error_grouped = error_csv.groupby('county').agg('mean').reset_index()
        error_grouped = error_grouped.sort_values(by='county')[['county', 'mse_base_pret', 'mse_addi_fine']]
        exec(f"error_grouped = pd.DataFrame(county_{p}).merge(error_grouped, on='county', how='left')")
        error_grouped.columns = ['county', 'mse_base', 'mse_prop']
        for i in ['base', 'prop']:
            exec(f"error_grouped['bin_{i}'] = pd.qcut(error_grouped['mse_{i}'], q=2, labels=False)")
            exec(f"error_grouped['bin_{i}'] = error_grouped['bin_{i}'].fillna(2)")
        error_grouped = error_grouped[['county', 'bin_base', 'bin_prop']]
        exec(f"error_grouped.to_csv('../results/book/s6/{model}_{p}_error_mapping.csv', index=False)")